# SWaT — Part 3: Real-Time Attack Detection System
### Live Modbus TCP → Feature Extraction → ML Inference → Alert

**Architecture:**
```
CODESYS PLC (192.168.5.194:1502)
        │  Modbus TCP (FC3 + FC1)
        ▼
  ModbusPoller  ──► SlidingWindowBuffer (seq_len=25)
        │                    │
        ▼                    ▼
  FeatureExtractor     SequenceExtractor
  (rolling/CUSUM)      (for LSTM)
        │                    │
        └──────┬─────────────┘
               ▼
      RealTimeDetector.predict()
         ├── XGBoost (tabular, fast)
         ├── MLP (tabular)
         └── BiLSTM (sequential)
               │
               ▼
       AlertManager → console / log / dashboard
```


## 0 · Imports & Load Models

In [ ]:
# !pip install pymodbus scikit-learn xgboost joblib torch numpy pandas

import time, threading, queue, logging, json as _json
from pathlib import Path
from collections import deque
from datetime import datetime

import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler("realtime_detection.log"),
        logging.StreamHandler()
    ]
)
log = logging.getLogger("SWaT-RT")

MODELS_DIR = Path("saved_models")
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Attack names ──────────────────────────────────────────────────────────────
ATTACK_NAMES = {
    0:"Normal Operation", 1:"Tank Overflow Attack", 2:"Chemical Depletion Attack",
    3:"Membrane Damage Attack", 4:"pH Manipulation Attack", 5:"Slow Ramp Attack",
    6:"Reconnaissance Scan", 7:"Denial of Service", 8:"Replay Attack",
    9:"Valve Manipulation Attack",
}
ALERT_LEVELS = {0:'INFO', 1:'CRITICAL', 2:'HIGH', 3:'HIGH',
                4:'CRITICAL', 5:'HIGH', 6:'MEDIUM', 7:'MEDIUM',
                8:'MEDIUM', 9:'HIGH'}

log.info(f"Device: {DEVICE}")


In [ ]:
# ── Load saved bundle ─────────────────────────────────────────────────────────
bundle = joblib.load(MODELS_DIR / "best_model_bundle.joblib")

SCALER       = bundle['scaler']
FEATURE_COLS = bundle['feature_cols']
DEAD_COLS    = bundle['dead_cols']
TEMPORAL_SENSORS = bundle.get('temporal_sensors',
    ['LIT_101','LIT_301','LIT_401','AIT_202','FIT_101',
     'FIT_401','Acid_Tank_Level','Chlorine_Residual'])
WINDOW_SIZES = bundle.get('temporal_windows', [10, 25, 50])
NUM_CLASSES  = bundle['num_classes']

# Fast inference model: XGBoost (tree, no GPU needed)
xgb_model = joblib.load(MODELS_DIR / "xgb_multiclass.joblib")
rf_model  = joblib.load(MODELS_DIR / "rf_multiclass.joblib")

log.info(f"Models loaded. Features: {len(FEATURE_COLS)}")
log.info(f"Temporal sensors: {TEMPORAL_SENSORS}")


In [ ]:
# ── Re-create MLP + LSTM architecture for loading weights ─────────────────────
class SWaTMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, num_classes, dropout=0.3):
        super().__init__()
        layers, prev = [], input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev,h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)
    def predict_proba(self, X_np):
        self.eval()
        with torch.no_grad():
            t = torch.FloatTensor(X_np).to(DEVICE)
            return torch.softmax(self.net(t), dim=1).cpu().numpy()

class SWaTLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, num_classes,
                 dropout=0.3, bidirectional=True):
        super().__init__()
        self.D = 2 if bidirectional else 1
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True,
                            dropout=dropout if num_layers>1 else 0,
                            bidirectional=bidirectional)
        self.attn = nn.Linear(hidden_dim*self.D, 1)
        self.norm = nn.LayerNorm(hidden_dim*self.D)
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(hidden_dim*self.D, num_classes)
    def forward(self, x):
        out, _ = self.lstm(x)
        w   = torch.softmax(self.attn(out), dim=1)
        ctx = (w * out).sum(dim=1)
        return self.fc(self.drop(self.norm(ctx)))

# Load MLP weights
INPUT_DIM = len(FEATURE_COLS)
mlp_model = SWaTMLP(INPUT_DIM, [512,256,128,64], NUM_CLASSES).to(DEVICE)
try:
    mlp_model.load_state_dict(
        torch.load(MODELS_DIR/"mlp_multiclass_best.pt", map_location=DEVICE))
    mlp_model.eval()
    log.info("MLP weights loaded")
except Exception as e:
    log.warning(f"MLP weights not found: {e}")

# Load LSTM weights
SEQ_LEN = 25
lstm_model = SWaTLSTM(INPUT_DIM, 128, 2, NUM_CLASSES).to(DEVICE)
try:
    lstm_model.load_state_dict(
        torch.load(MODELS_DIR/"lstm_best.pt", map_location=DEVICE))
    lstm_model.eval()
    log.info("LSTM weights loaded")
except Exception as e:
    log.warning(f"LSTM weights not found: {e}")


## 1 · Modbus Poller

In [ ]:
from pymodbus.client import ModbusTcpClient

# ── Register map (from plant.st) ──────────────────────────────────────────────
HOLDING_REGS = {
    'FIT_101':0,'LIT_101':1,'MV_101':2,
    'AIT_201':3,'AIT_202':4,'AIT_203':5,'FIT_201':6,
    'Acid_Tank_Level':7,'Chlorine_Tank_Level':8,
    'Coagulant_Tank_Level':9,'Bisulfate_Tank_Level':10,
    'DPIT_301':12,'FIT_301':13,'LIT_301':14,
    'AIT_401':23,'AIT_402':24,'FIT_401':25,'LIT_401':26,
    'AIT_501':28,'FIT_501':31,
    'PIT_501':35,'PIT_502':36,'PIT_503':37,
    'TDS_Feed':41,'TDS_Permeate':42,
    'FIT_601':43,'Turbidity_Raw':44,
    'Chlorine_Residual':51,
}
# Coil map (digital outputs)
COIL_MAP = {
    'P_101':0,'P_102':1,'P_203':2,'P_205':3,'P_206':4,
    'MV_201':5,'P_301':6,'MV_301':7,'MV_302':8,
    'P_401':9,'P_403':10,
    'P_501':11,
    'P_601':12,'P_602':13,'P_603':14,
    'Chemical_Low_Alarm':20,'High_Level_Alarm':21,'System_Run':23,
}

class ModbusPoller:
    """
    Polls PLC over Modbus TCP.
    Reads 52 holding registers + 25 coils in 2 round-trips.
    Returns a dict of sensor_name -> value.
    """
    def __init__(self, host='192.168.5.194', port=1502, unit_id=1,
                 timeout=1.0, retries=3):
        self.host    = host
        self.port    = port
        self.unit_id = unit_id
        self.timeout = timeout
        self.retries = retries
        self.client  = None
        self._lock   = threading.Lock()

    def connect(self):
        self.client = ModbusTcpClient(self.host, port=self.port,
                                      timeout=self.timeout)
        ok = self.client.connect()
        if ok:
            log.info(f"Modbus connected: {self.host}:{self.port}")
        else:
            raise ConnectionError(f"Cannot connect to {self.host}:{self.port}")
        return ok

    def disconnect(self):
        if self.client:
            self.client.close()
            log.info("Modbus disconnected")

    def _read_registers(self) -> dict:
        """Bulk read: 2 Modbus calls total."""
        result = {}
        with self._lock:
            # FC3 — 52 holding registers (all analogue sensors)
            rr = self.client.read_holding_registers(0, 52, unit=self.unit_id)
            if rr.isError():
                raise IOError(f"FC3 error: {rr}")
            regs = rr.registers
            for name, addr in HOLDING_REGS.items():
                if addr < len(regs):
                    val = regs[addr]
                    # AIT_202 is stored ×100 in PLC
                    if name == 'AIT_202':
                        val = val / 100.0
                    result[name] = float(val)

            # FC1 — 25 coils (digital I/O)
            rc = self.client.read_coils(0, 25, unit=self.unit_id)
            if rc.isError():
                raise IOError(f"FC1 error: {rc}")
            bits = rc.bits
            for name, addr in COIL_MAP.items():
                if addr < len(bits):
                    result[name] = int(bits[addr])
        return result

    def poll(self) -> dict | None:
        """Poll with retry logic. Returns None on persistent failure."""
        for attempt in range(self.retries):
            try:
                data = self._read_registers()
                data['timestamp'] = datetime.utcnow().isoformat()
                return data
            except Exception as e:
                log.warning(f"Poll attempt {attempt+1}/{self.retries} failed: {e}")
                if attempt < self.retries - 1:
                    time.sleep(0.05)
                else:
                    log.error("All poll attempts failed")
                    return None


## 2 · Sliding Window Buffer & Feature Extractor

In [ ]:
class SlidingWindowBuffer:
    """
    Thread-safe circular buffer.
    Accumulates raw poll results; provides windows for feature extraction.
    """
    def __init__(self, maxlen=200):
        self._buf = deque(maxlen=maxlen)
        self._lock = threading.Lock()

    def push(self, reading: dict):
        with self._lock:
            self._buf.append(reading)

    def get_window(self, n: int) -> list:
        """Return the last n readings as a list of dicts."""
        with self._lock:
            buf = list(self._buf)
        return buf[-n:] if len(buf) >= n else buf

    def __len__(self):
        with self._lock:
            return len(self._buf)


class FeatureExtractor:
    """
    Converts a window of raw poll dicts into a single feature vector
    matching FEATURE_COLS (same pipeline as training).
    """
    def __init__(self, feature_cols, temporal_sensors,
                 window_sizes, scaler, dead_cols):
        self.feature_cols      = feature_cols
        self.temporal_sensors  = temporal_sensors
        self.window_sizes      = window_sizes
        self.scaler            = scaler
        self.dead_cols         = set(dead_cols)
        # Per-run-start baselines for tank normalisation
        self._tank_starts: dict = {}
        self._cusum_state: dict = {}   # {sensor: last_cusum_value}
        self._run_means:   dict = {}   # {sensor: expanding_mean}
        self._n_samples    = 0

    def reset(self):
        """Call when starting a new detection session."""
        self._tank_starts = {}
        self._cusum_state = {}
        self._run_means   = {}
        self._n_samples   = 0

    def _update_cusum(self, sensor: str, value: float) -> float:
        """Incremental CUSUM update."""
        n = self._n_samples + 1
        prev_mean = self._run_means.get(sensor, value)
        new_mean  = prev_mean + (value - prev_mean) / n
        self._run_means[sensor] = new_mean
        deviation = value - new_mean
        prev_cusum = self._cusum_state.get(sensor, 0.0)
        new_cusum  = prev_cusum + deviation
        self._cusum_state[sensor] = new_cusum
        return new_cusum

    def extract(self, window: list) -> np.ndarray | None:
        """
        window: list of raw poll dicts (most recent last).
        Returns scaled feature vector of shape (1, n_features) or None.
        """
        if len(window) < 2:
            return None

        # Build DataFrame from window
        df_win = pd.DataFrame(window)

        # Drop dead columns
        drop = [c for c in self.dead_cols if c in df_win.columns]
        df_win = df_win.drop(columns=drop, errors='ignore')

        # Bool -> int
        for c in df_win.select_dtypes(include=bool).columns:
            df_win[c] = df_win[c].astype(np.int8)

        # Tank normalisation
        TANK_COLS = ['Acid_Tank_Level','Chlorine_Tank_Level',
                     'Coagulant_Tank_Level','Bisulfate_Tank_Level']
        for col in TANK_COLS:
            if col not in df_win.columns: continue
            if col not in self._tank_starts:
                self._tank_starts[col] = df_win[col].iloc[0]
            start = self._tank_starts[col]
            df_win[f'{col}_pct'] = ((start - df_win[col]) /
                                    max(start, 1)).clip(0, 1)

        # Temporal features on the window
        dt = pd.Series([0.2] * len(df_win))   # nominal poll interval
        for s in self.temporal_sensors:
            if s not in df_win.columns: continue
            col_v = df_win[s].astype(float)
            df_win[f'd_{s}_dt'] = col_v.diff().fillna(0) / dt
            for w in self.window_sizes:
                df_win[f'{s}_rmean{w}'] = col_v.rolling(w, min_periods=1).mean()
                df_win[f'{s}_rstd{w}']  = col_v.rolling(w, min_periods=1).std().fillna(0)
            # CUSUM from last row (incremental)
            last_val   = float(col_v.iloc[-1])
            cusum_val  = self._update_cusum(s, last_val)
            df_win[f'{s}_cusum'] = np.nan   # placeholder
            df_win.loc[df_win.index[-1], f'{s}_cusum'] = cusum_val
            df_win[f'{s}_cusum'] = df_win[f'{s}_cusum'].ffill().fillna(0)

        self._n_samples += 1

        # Build final feature row (last row = most recent)
        last = df_win.iloc[[-1]].copy()

        # Align to training feature columns
        feat_vec = np.zeros((1, len(self.feature_cols)), dtype=np.float32)
        for i, col in enumerate(self.feature_cols):
            if col in last.columns:
                v = last[col].values[0]
                feat_vec[0, i] = 0.0 if (pd.isna(v) or np.isinf(v)) else float(v)

        # Scale
        feat_scaled = self.scaler.transform(feat_vec)
        return feat_scaled.astype(np.float32)


## 3 · Ensemble Detector

In [ ]:
class EnsembleDetector:
    """
    Runs XGBoost + MLP (tabular) + LSTM (sequence) in parallel.
    Combines probabilities via weighted average.
    """
    def __init__(self, xgb_model, mlp_model, lstm_model,
                 seq_len=25, device=DEVICE,
                 weights=(0.4, 0.35, 0.25)):
        self.xgb    = xgb_model
        self.mlp    = mlp_model
        self.lstm   = lstm_model
        self.seq_len = seq_len
        self.device  = device
        self.w_xgb, self.w_mlp, self.w_lstm = weights
        # Sequence buffer for LSTM
        self._seq_buf = deque(maxlen=seq_len)

    def _lstm_predict(self, feat_vec: np.ndarray) -> np.ndarray | None:
        """Returns class probabilities from LSTM or None if buffer not full."""
        self._seq_buf.append(feat_vec[0])   # (n_features,)
        if len(self._seq_buf) < self.seq_len:
            return None
        seq = np.array(self._seq_buf, dtype=np.float32)    # (seq_len, n_features)
        seq_t = torch.FloatTensor(seq).unsqueeze(0).to(self.device)  # (1, T, F)
        self.lstm.eval()
        with torch.no_grad():
            logits = self.lstm(seq_t)
            proba  = torch.softmax(logits, dim=1).cpu().numpy()[0]
        return proba

    def predict(self, feat_vec: np.ndarray) -> dict:
        """
        feat_vec: (1, n_features) scaled array
        Returns:
            {
              'class_id': int,
              'class_name': str,
              'confidence': float,
              'probabilities': np.ndarray,
              'model_votes': dict,
              'is_attack': bool,
              'alert_level': str,
            }
        """
        # XGBoost
        xgb_proba = self.xgb.predict_proba(feat_vec)[0]

        # MLP
        mlp_proba = self.mlp.predict_proba(feat_vec)[0]

        # LSTM
        lstm_proba = self._lstm_predict(feat_vec)

        # Weighted ensemble
        if lstm_proba is not None:
            proba = (self.w_xgb * xgb_proba +
                     self.w_mlp * mlp_proba +
                     self.w_lstm * lstm_proba)
        else:
            # Fall back to XGB + MLP while LSTM buffer fills
            w_sum = self.w_xgb + self.w_mlp
            proba = ((self.w_xgb/w_sum) * xgb_proba +
                     (self.w_mlp/w_sum)  * mlp_proba)

        class_id = int(np.argmax(proba))
        return {
            'class_id':    class_id,
            'class_name':  ATTACK_NAMES[class_id],
            'confidence':  float(proba[class_id]),
            'probabilities': proba.tolist(),
            'model_votes': {
                'xgb_pred':  int(np.argmax(xgb_proba)),
                'mlp_pred':  int(np.argmax(mlp_proba)),
                'lstm_pred': int(np.argmax(lstm_proba)) if lstm_proba is not None else None,
            },
            'is_attack':   class_id > 0,
            'alert_level': ALERT_LEVELS[class_id],
        }

    def reset(self):
        self._seq_buf.clear()


detector = EnsembleDetector(xgb_model, mlp_model, lstm_model, seq_len=SEQ_LEN)
extractor = FeatureExtractor(FEATURE_COLS, TEMPORAL_SENSORS,
                              WINDOW_SIZES, SCALER, DEAD_COLS)
log.info("Ensemble detector ready")


## 4 · Alert Manager

In [ ]:
class AlertManager:
    """
    Manages alert state, suppresses repeated identical alerts,
    writes to log file and alert history CSV.
    """
    SUPPRESS_SECS = 10   # don't repeat same alert within 10 seconds
    ALERT_COLORS  = {
        'CRITICAL': '\033[91m',   # red
        'HIGH':     '\033[93m',   # yellow
        'MEDIUM':   '\033[94m',   # blue
        'INFO':     '\033[92m',   # green
        'RESET':    '\033[0m'
    }

    def __init__(self, log_csv='alert_history.csv', conf_threshold=0.55):
        self.log_csv        = Path(log_csv)
        self.conf_threshold = conf_threshold
        self._last_alert    = {}   # {class_id: last_alert_time}
        self._alert_count   = 0
        self._fp_suppressed = 0
        self._history       = []

        # Write CSV header
        if not self.log_csv.exists():
            with open(self.log_csv, 'w') as f:
                f.write("timestamp,class_id,class_name,confidence,alert_level,run_id
")

    def process(self, result: dict, raw_reading: dict, run_id: int = 0):
        cid   = result['class_id']
        conf  = result['confidence']
        level = result['alert_level']
        ts    = raw_reading.get('timestamp', datetime.utcnow().isoformat())

        # ── Suppress low-confidence predictions ──────────────────────────────
        if conf < self.conf_threshold and cid != 0:
            self._fp_suppressed += 1
            return

        # ── Suppress repeated same-class alerts ──────────────────────────────
        now = time.time()
        if cid != 0:
            last = self._last_alert.get(cid, 0)
            if now - last < self.SUPPRESS_SECS:
                return
            self._last_alert[cid] = now

        # ── Format and print ──────────────────────────────────────────────────
        col = self.ALERT_COLORS.get(level, '')
        rst = self.ALERT_COLORS['RESET']
        icon = {'CRITICAL':'🚨','HIGH':'⚠️','MEDIUM':'ℹ️','INFO':'✅'}.get(level,'')
        print(f"{col}[{ts}] {icon}  [{level}]  "
              f"{result['class_name']}  (conf={conf:.3f}){rst}")

        if cid != 0:
            log.warning(f"ATTACK DETECTED: {result['class_name']} "
                        f"conf={conf:.3f} level={level}")
            self._alert_count += 1

        # ── Write to CSV ──────────────────────────────────────────────────────
        with open(self.log_csv, 'a') as f:
            f.write(f"{ts},{cid},{result['class_name']},"
                    f"{conf:.4f},{level},{run_id}\n")

        self._history.append({**result, 'timestamp': ts, 'run_id': run_id})

    def stats(self):
        print(f"\n=== Alert Stats ===")
        print(f"  Alerts fired     : {self._alert_count}")
        print(f"  Low-conf filtered: {self._fp_suppressed}")
        return self._history

alert_mgr = AlertManager(conf_threshold=0.55)
log.info("Alert manager ready")


## 5 · Real-Time Detection Loop

In [ ]:
class RealTimeDetector:
    """
    Main orchestration loop.
    Spawns a polling thread; processes readings on the main thread.
    """
    def __init__(self, poller, buffer, extractor,
                 detector, alert_mgr,
                 poll_interval=0.2, window_size=50):
        self.poller        = poller
        self.buffer        = buffer
        self.extractor     = extractor
        self.detector      = detector
        self.alert_mgr     = alert_mgr
        self.poll_interval = poll_interval
        self.window_size   = window_size
        self._running      = threading.Event()
        self._poll_q       = queue.Queue(maxsize=100)
        self._stats        = {
            'polls': 0, 'errors': 0, 'predictions': 0,
            'attacks_detected': 0, 'start_time': None
        }

    # ── Background polling thread ─────────────────────────────────────────────
    def _poll_thread(self):
        log.info("Poll thread started")
        while self._running.is_set():
            t0 = time.perf_counter()
            reading = self.poller.poll()
            if reading:
                self._poll_q.put(reading, block=False)
                self._stats['polls'] += 1
            else:
                self._stats['errors'] += 1
            # Maintain target poll rate
            elapsed = time.perf_counter() - t0
            sleep_t = max(0, self.poll_interval - elapsed)
            time.sleep(sleep_t)

    # ── Main processing loop ─────────────────────────────────────────────────
    def run(self, duration_s=None, verbose_every=50):
        """
        duration_s: run for this many seconds (None = run until stop() called)
        verbose_every: print stats every N predictions
        """
        self._running.set()
        self._stats['start_time'] = time.time()
        extractor.reset(); detector.reset()

        # Start poll thread
        poll_t = threading.Thread(target=self._poll_thread, daemon=True)
        poll_t.start()

        log.info(f"Detection loop started  (poll={self.poll_interval}s  "
                 f"window={self.window_size})")
        try:
            while self._running.is_set():
                # Time-box the run
                if duration_s and (time.time()-self._stats['start_time']) >= duration_s:
                    break

                # Drain queue
                try:
                    reading = self._poll_q.get(timeout=1.0)
                except queue.Empty:
                    continue

                self.buffer.push(reading)

                # Need at least min-window samples before extracting features
                if len(self.buffer) < max(self.window_size // 2, 10):
                    continue

                window    = self.buffer.get_window(self.window_size)
                feat_vec  = self.extractor.extract(window)
                if feat_vec is None:
                    continue

                result    = self.detector.predict(feat_vec)
                self._stats['predictions'] += 1
                if result['is_attack']:
                    self._stats['attacks_detected'] += 1

                self.alert_mgr.process(result, reading)

                # Periodic stats print
                if self._stats['predictions'] % verbose_every == 0:
                    elapsed = time.time() - self._stats['start_time']
                    rate    = self._stats['polls'] / max(elapsed, 1)
                    log.info(f"[{elapsed:.0f}s] polls={self._stats['polls']} "
                             f"preds={self._stats['predictions']} "
                             f"attacks={self._stats['attacks_detected']} "
                             f"rate={rate:.1f}Hz")

        except KeyboardInterrupt:
            log.info("Keyboard interrupt — stopping")
        finally:
            self.stop()

    def stop(self):
        self._running.clear()
        log.info("Detection loop stopped")
        self.alert_mgr.stats()
        total = time.time() - (self._stats['start_time'] or time.time())
        log.info(f"Final stats: {self._stats} | total={total:.1f}s")


## 6 · Run Live Detection (PLC Connected)

In [ ]:
# ── Step-by-step: connect to real PLC and run ─────────────────────────────────

PLC_HOST = '192.168.5.194'
PLC_PORT = 1502
UNIT_ID  = 1

poller = ModbusPoller(host=PLC_HOST, port=PLC_PORT,
                      unit_id=UNIT_ID, timeout=1.0, retries=3)
buffer = SlidingWindowBuffer(maxlen=200)

rt_detector = RealTimeDetector(
    poller        = poller,
    buffer        = buffer,
    extractor     = extractor,
    detector      = detector,
    alert_mgr     = alert_mgr,
    poll_interval = 0.2,   # 5 Hz
    window_size   = 50,    # 10-second feature window
)

try:
    poller.connect()
    print("PLC connected — starting detection loop (Ctrl+C to stop)")
    rt_detector.run(duration_s=None)   # run indefinitely
except ConnectionError as e:
    print(f"Cannot connect to PLC: {e}")
    print("Use the SIMULATION MODE cell below instead.")
finally:
    poller.disconnect()


## 7 · Simulation Mode (No PLC Required)

In [ ]:
"""
Replays the saved CSV through the full detection pipeline.
Useful for testing without a live PLC connection.
"""

import pandas as pd

class CSVReplayPoller:
    """Replays a CSV file at the given speed multiplier."""
    def __init__(self, csv_path, speed=5.0, loop=False):
        df = pd.read_csv(csv_path, compression='gzip'
                         if str(csv_path).endswith('.gz') else None)
        df['ATTACK_ID'] = df['ATTACK_ID'].map(
            {0:0,8:1,9:2,10:3,11:4,12:5,13:6,14:7,15:8,16:9})
        self._rows   = df.to_dict('records')
        self._idx    = 0
        self._speed  = speed
        self._loop   = loop
        self._gt_labels = []   # ground-truth for accuracy calc

    def connect(self): log.info(f"CSV replay poller ready ({len(self._rows):,} rows)")
    def disconnect(self): log.info("CSV replay finished")

    def poll(self):
        if self._idx >= len(self._rows):
            if self._loop:
                self._idx = 0
            else:
                return None
        row = self._rows[self._idx].copy()
        self._gt_labels.append(int(row.get('ATTACK_ID', 0)))
        row['timestamp'] = datetime.utcnow().isoformat()
        self._idx += 1
        time.sleep(0.2 / self._speed)
        return row

    @property
    def ground_truth(self): return self._gt_labels


# ── Run simulation ────────────────────────────────────────────────────────────
csv_poller  = CSVReplayPoller("compressed_data_csv.gz", speed=20.0)
sim_buffer  = SlidingWindowBuffer(maxlen=200)
sim_extractor = FeatureExtractor(FEATURE_COLS, TEMPORAL_SENSORS,
                                  WINDOW_SIZES, SCALER, DEAD_COLS)
sim_detector  = EnsembleDetector(xgb_model, mlp_model, lstm_model, seq_len=SEQ_LEN)
sim_alerts    = AlertManager(log_csv='sim_alerts.csv', conf_threshold=0.55)

sim_rt = RealTimeDetector(
    poller        = csv_poller,
    buffer        = sim_buffer,
    extractor     = sim_extractor,
    detector      = sim_detector,
    alert_mgr     = sim_alerts,
    poll_interval = 0.0,   # no sleep — go as fast as possible
    window_size   = 50,
)

csv_poller.connect()
print("Starting simulation (replaying CSV at 20x speed, 60s window)...")
sim_rt.run(duration_s=60, verbose_every=100)
csv_poller.disconnect()


## 8 · Simulation Results & Accuracy

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

history = sim_alerts._history

if history:
    pred_ids  = [h['class_id']   for h in history]
    confs     = [h['confidence'] for h in history]
    timestamps = [h['timestamp'] for h in history]

    # Align ground truth to predictions (predictions start after buffer fills)
    gt = csv_poller.ground_truth
    offset = len(gt) - len(pred_ids)
    gt_aligned = gt[max(0, offset):offset+len(pred_ids)]
    min_len = min(len(pred_ids), len(gt_aligned))
    pred_ids  = pred_ids[:min_len]
    gt_aligned = gt_aligned[:min_len]
    confs      = confs[:min_len]

    print("=== Simulation Detection Report ===")
    print(classification_report(
        gt_aligned, pred_ids,
        target_names=[ATTACK_NAMES[i] for i in range(NUM_CLASSES)],
        labels=list(range(NUM_CLASSES)), digits=4, zero_division=0))

    # ── Plots ─────────────────────────────────────────────────────────────────
    PALETTE = ['#1565C0','#E53935','#2E7D32','#F57F17','#6A1B9A',
               '#00838F','#AD1457','#37474F','#558B2F','#4E342E']

    fig = plt.figure(figsize=(16, 14))
    gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.5, wspace=0.35)

    # 1. Ground truth vs prediction timeline
    ax1 = fig.add_subplot(gs[0, :])
    x   = range(len(pred_ids))
    ax1.step(x, gt_aligned, where='post', color='#212121', lw=1, label='Ground Truth', alpha=0.7)
    ax1.step(x, pred_ids,   where='post', color='#E53935',  lw=1, label='Predicted',    alpha=0.7, ls='--')
    ax1.set_title("Ground Truth vs Predicted Attack Class — Simulation",
                  fontweight='bold')
    ax1.set_ylabel("Class ID"); ax1.set_xlabel("Sample")
    ax1.legend(); ax1.grid(alpha=0.3)
    ax1.set_yticks(range(NUM_CLASSES))
    ax1.set_yticklabels([f"[{i}] {ATTACK_NAMES[i][:14]}" for i in range(NUM_CLASSES)],
                        fontsize=7)

    # 2. Confidence over time
    ax2 = fig.add_subplot(gs[1, 0])
    attack_mask = np.array(pred_ids) > 0
    ax2.plot(range(len(confs)), confs, lw=0.8, color='grey', alpha=0.5)
    ax2.scatter(np.where(~attack_mask)[0], np.array(confs)[~attack_mask],
                s=4, color='#1565C0', alpha=0.4, label='Normal')
    ax2.scatter(np.where(attack_mask)[0],  np.array(confs)[attack_mask],
                s=6, color='#E53935', alpha=0.6, label='Attack')
    ax2.axhline(0.55, color='orange', ls='--', lw=1.2, label='Threshold=0.55')
    ax2.set_title("Detection Confidence over Time", fontweight='bold')
    ax2.set_ylabel("Confidence"); ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

    # 3. Confusion matrix
    ax3 = fig.add_subplot(gs[1, 1])
    cm = confusion_matrix(gt_aligned, pred_ids, normalize='true',
                          labels=list(range(NUM_CLASSES)))
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=[f"[{i}]" for i in range(NUM_CLASSES)],
                yticklabels=[f"[{i}]" for i in range(NUM_CLASSES)],
                ax=ax3, linewidths=0.4, annot_kws={'size':7})
    ax3.set_title("Simulation Confusion Matrix", fontweight='bold')
    ax3.tick_params(labelsize=7)

    # 4. Attack detection rate
    ax4 = fig.add_subplot(gs[2, 0])
    from collections import Counter
    gt_counts   = Counter(gt_aligned)
    correct_map = {}
    for true, pred in zip(gt_aligned, pred_ids):
        if true not in correct_map: correct_map[true] = {'correct':0,'total':0}
        correct_map[true]['total']   += 1
        correct_map[true]['correct'] += int(true == pred)
    classes_present = sorted(correct_map.keys())
    recall_vals = [correct_map[c]['correct']/max(correct_map[c]['total'],1)
                   for c in classes_present]
    ax4.barh([f"[{c}] {ATTACK_NAMES[c][:16]}" for c in classes_present],
             recall_vals, color=[PALETTE[c] for c in classes_present])
    ax4.axvline(0.8, color='red', ls='--', lw=1.2, label='80% target')
    ax4.set_xlim(0, 1.1); ax4.set_title("Per-class Recall", fontweight='bold')
    ax4.legend(fontsize=8); ax4.grid(axis='x', alpha=0.3)

    # 5. Alert timeline (attacks only)
    ax5 = fig.add_subplot(gs[2, 1])
    alert_times = [i for i,p in enumerate(pred_ids) if p > 0]
    alert_class = [pred_ids[i] for i in alert_times]
    ax5.scatter(alert_times, alert_class,
                c=[PALETTE[c] for c in alert_class], s=8, alpha=0.7)
    ax5.set_title("Alert Timeline (predicted attacks only)", fontweight='bold')
    ax5.set_ylabel("Class ID"); ax5.set_xlabel("Sample")
    ax5.set_yticks(range(1, NUM_CLASSES))
    ax5.set_yticklabels([ATTACK_NAMES[i][:14] for i in range(1, NUM_CLASSES)], fontsize=7)
    ax5.grid(alpha=0.3)

    plt.suptitle("Real-Time Simulation Results", fontweight='bold', fontsize=14)
    plt.savefig("plots/15_simulation_results.png", dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No predictions recorded — run simulation cell first.")


## 9 · Live Dashboard (Optional — requires ipywidgets)

In [ ]:
# Run this cell in Jupyter to get a live updating dashboard
# Requires: pip install ipywidgets

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not installed — skipping dashboard")

if HAS_WIDGETS:
    # State shared between threads
    _dash_state = {'history': [], 'running': False}

    out_main  = widgets.Output()
    btn_start = widgets.Button(description='Start Detection', button_style='success')
    btn_stop  = widgets.Button(description='Stop',            button_style='danger')
    lbl_status = widgets.Label("Status: idle")

    PALETTE_HEX = ['#1565C0','#E53935','#2E7D32','#F57F17','#6A1B9A',
                   '#00838F','#AD1457','#37474F','#558B2F','#4E342E']

    def _update_dashboard(result, reading):
        cid  = result['class_id']
        conf = result['confidence']
        _dash_state['history'].append(result)
        hist = _dash_state['history'][-200:]

        with out_main:
            clear_output(wait=True)
            fig, axes = plt.subplots(1, 3, figsize=(16, 4))

            # Live prediction bar
            proba = result['probabilities']
            axes[0].barh([ATTACK_NAMES[i][:18] for i in range(NUM_CLASSES)],
                         proba, color=PALETTE_HEX)
            axes[0].axvline(0.55, color='red', ls='--', lw=1)
            axes[0].set_title(f"Current Prediction: [{cid}] {ATTACK_NAMES[cid]}\n"
                              f"Confidence: {conf:.3f}", fontweight='bold',
                              color='red' if cid>0 else 'green')
            axes[0].set_xlim(0,1)

            # Recent class timeline
            recent_ids = [h['class_id'] for h in hist]
            axes[1].step(range(len(recent_ids)), recent_ids, where='post', lw=1)
            axes[1].set_title("Recent Predictions (last 200)", fontweight='bold')
            axes[1].set_yticks(range(NUM_CLASSES))
            axes[1].set_yticklabels([f"[{i}]" for i in range(NUM_CLASSES)], fontsize=7)
            axes[1].grid(alpha=0.3)

            # Confidence timeline
            recent_conf = [h['confidence'] for h in hist]
            recent_atk  = [h['is_attack']  for h in hist]
            axes[2].plot(range(len(recent_conf)), recent_conf, lw=0.8, color='grey')
            ax_atk = [i for i,a in enumerate(recent_atk) if a]
            axes[2].scatter(ax_atk, [recent_conf[i] for i in ax_atk],
                            s=8, color='red', zorder=5)
            axes[2].axhline(0.55, color='orange', ls='--', lw=1)
            axes[2].set_title("Confidence (red=attack)", fontweight='bold')
            axes[2].set_ylim(0,1); axes[2].grid(alpha=0.3)

            plt.tight_layout(); plt.show()

    # Monkey-patch alert manager to also update dashboard
    _orig_process = alert_mgr.process
    def _patched_process(result, reading, run_id=0):
        _orig_process(result, reading, run_id)
        if _dash_state['running']:
            _update_dashboard(result, reading)
    alert_mgr.process = _patched_process

    def on_start(b):
        _dash_state['running'] = True
        lbl_status.value = "Status: running"
        csv_p = CSVReplayPoller("compressed_data_csv.gz", speed=10.0)
        csv_p.connect()
        rt2 = RealTimeDetector(csv_p, SlidingWindowBuffer(),
                                FeatureExtractor(FEATURE_COLS, TEMPORAL_SENSORS,
                                                  WINDOW_SIZES, SCALER, DEAD_COLS),
                                EnsembleDetector(xgb_model, mlp_model, lstm_model),
                                alert_mgr, poll_interval=0.0, window_size=50)
        t = threading.Thread(target=rt2.run, kwargs={'duration_s':120}, daemon=True)
        t.start()

    def on_stop(b):
        _dash_state['running'] = False
        lbl_status.value = "Status: stopped"

    btn_start.on_click(on_start)
    btn_stop.on_click(on_stop)

    display(widgets.VBox([
        widgets.HBox([btn_start, btn_stop, lbl_status]),
        out_main
    ]))


## 10 · Step-by-Step Guide

### Running against a live PLC
```
Step 1: Ensure CODESYS v3.5 runtime is running on 192.168.5.194:1502
Step 2: Run cells 0 → 5 (Imports through RealTimeDetector class)
Step 3: Run cell 6 (Live Detection)
        → Detection runs at 5 Hz
        → Ctrl+C to stop
        → Alerts written to realtime_detection.log + alert_history.csv
```

### Running in simulation (no PLC)
```
Step 1: Run cells 0 → 5
Step 2: Run cell 7 (Simulation Mode) — replays compressed_data_csv.gz at 20x speed
Step 3: Run cell 8 to see accuracy metrics and plots
```

### Tuning the detector
| Parameter | Where | Effect |
|-----------|-------|--------|
| `conf_threshold=0.55` | AlertManager | Raise to reduce false alarms, lower to catch more |
| `SUPPRESS_SECS=10` | AlertManager | Time between repeated same-class alerts |
| `weights=(0.4,0.35,0.25)` | EnsembleDetector | XGB/MLP/LSTM blend weights |
| `poll_interval=0.2` | RealTimeDetector | Poll rate (0.1 = 10Hz) |
| `window_size=50` | RealTimeDetector | Feature extraction window (10s at 5Hz) |
| `SEQ_LEN=25` | EnsembleDetector | LSTM sequence window (5s at 5Hz) |

### Alert levels
| Level | Attacks |
|-------|---------|
| CRITICAL | Tank Overflow, pH Manipulation |
| HIGH | Chemical Depletion, Slow Ramp, Membrane Damage, Valve Manipulation |
| MEDIUM | Reconnaissance, DoS, Replay |
| INFO | Normal Operation |
